# 06 · Primera llegada sustantiva y recompensa temporal — Bogotá 2023

**Objetivo.** Estimar accesibilidad estocástica desde cada UTAM hacia destinos revelados de trabajo, educación y salud, comparando las definiciones `core_50` y `extended_80` aceptadas por la Compuerta 6.

Se reportan dos magnitudes distintas:

1. **Tiempo de primera llegada en transiciones:** número de pasos de la cadena hasta alcanzar el conjunto objetivo.
2. **Minutos acumulados esperados:** recompensa temporal obtenida de la duración media ponderada de cada transición.

La segunda magnitud es una expectativa de un **Markov reward process**. No reconstruye trayectorias individuales ni equivale a tiempo mínimo de viaje.


In [ ]:
%pip -q install pandas pyarrow numpy scipy matplotlib seaborn


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import re
import shutil
import unicodedata
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate7"
OUT = ROOT / "outputs/bogota_em2023/gate7"
DRIVE_BASE = Path(
    "/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023"
)
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = (
    "https://observatorio.movilidadbogota.gov.co/sites/default/files/"
    "2025-09/05_Base%20datos%20procesada%20EODH.zip"
)
PURPOSES = ("employment", "education", "health")
TARGET_DEFINITIONS = ("core_50", "extended_80")
QUANTILE_HORIZON = 1000

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Entorno fuera de Colab o Drive no montado: {exc}")

gate4_dir = Path(os.environ.get("GATE4_DIR_OVERRIDE", DRIVE_BASE / "results/gate4"))
gate5_dir = Path(os.environ.get("GATE5_DIR_OVERRIDE", DRIVE_BASE / "results/gate5"))
gate6_dir = Path(os.environ.get("GATE6_DIR_OVERRIDE", DRIVE_BASE / "results/gate6"))

EDGE_PATH = gate4_dir / "transition_edges_utam.parquet"
GATE5_PATH = gate5_dir / "gate_5.json"
GATE6_PATH = gate6_dir / "gate_6.json"
TARGET_PATH = gate6_dir / "purpose_target_sets.csv"

required_paths = (EDGE_PATH, GATE5_PATH, GATE6_PATH, TARGET_PATH)
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan insumos validados: " + ", ".join(missing))

gate5 = json.loads(GATE5_PATH.read_text(encoding="utf-8"))
gate6 = json.loads(GATE6_PATH.read_text(encoding="utf-8"))
gate5_status = gate5.get("gate", gate5)
gate6_status = gate6.get("gate", gate6)
if not bool(gate5_status.get("gate_5_passed", False)):
    raise RuntimeError("La Compuerta 5 no fue aprobada.")
if not bool(gate6_status.get("gate_6_passed", False)):
    raise RuntimeError("La Compuerta 6 no fue aprobada.")

print("Kernel:", EDGE_PATH)
print("Destinos:", TARGET_PATH)
print("Salida:", OUT)


## 1. Recuperación de duraciones

Se reutiliza el ZIP oficial de la encuesta almacenado en Drive. El notebook no intenta descargarlo automáticamente desde el portal. Las duraciones se agregan por par origen–destino usando `fexp_vj`.


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)


def upload_archive(destination):
    from google.colab import files

    print("Descargue el ZIP en el navegador si no lo conserva:")
    print(SOURCE_URL)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Suba exactamente un archivo ZIP.")
    name, content = next(iter(uploaded.items()))
    temporary = destination.with_suffix(".zip.upload")
    temporary.write_bytes(content)
    del content
    if not valid_zip(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError(f"{name} no es un ZIP válido.")
    temporary.replace(destination)
    return "manual_upload"


archive = RAW / "processed_edoh.zip"
drive_archive = DRIVE_BASE / "processed_edoh.zip"
if valid_zip(archive):
    retrieval_method = "runtime_cache"
elif valid_zip(drive_archive):
    shutil.copy2(drive_archive, archive)
    retrieval_method = "google_drive_cache"
else:
    retrieval_method = upload_archive(archive)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, drive_archive)


def extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    queue = [(Path(zip_path), destination)]
    visited = set()
    while queue:
        current, target = queue.pop(0)
        signature = (str(current.resolve()), str(target.resolve()))
        if signature in visited:
            continue
        visited.add(signature)
        with zipfile.ZipFile(current) as zipped:
            zipped.extractall(target)
        for nested in target.rglob("*.zip"):
            nested_target = nested.parent / nested.stem
            nested_signature = (str(nested.resolve()), str(nested_target.resolve()))
            if nested_signature not in visited:
                queue.append((nested, nested_target))


extract_recursive(archive, INTERIM / "processed_edoh")
source_manifest = pd.DataFrame([{
    "source_id": "processed_edoh",
    "url": SOURCE_URL,
    "path": str(archive),
    "bytes": int(archive.stat().st_size),
    "sha256": sha256(archive),
    "retrieval_method": retrieval_method,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}])
source_manifest.to_csv(OUT / "source_manifest.csv", index=False)
display(source_manifest)


## 2. Kernel y recompensas por transición

El kernel (P) se reconstruye exactamente desde Gate 4. Para cada arista se estima su duración media ponderada. Si una arista carece de duración, se imputa primero la media del origen y, excepcionalmente, la media global; toda imputación queda auditada.


In [ ]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def clean_key(series):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.str.replace(r"\.0$", "", regex=True)
    invalid = cleaned.map(normalize).isin(
        ["", "nan", "none", "na", "no_aplica", "sin_informacion"]
    )
    return cleaned.mask(invalid)


def canonical_utam(series):
    cleaned = clean_key(series)

    def canonical(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper().replace(" ", "")
        upr = re.fullmatch(r"UPR0*(\d+)", text)
        if upr:
            return f"UPR{int(upr.group(1)):04d}"
        utam = re.fullmatch(r"(?:UTAM)?0*(\d+)", text)
        if utam:
            return f"UTAM{int(utam.group(1)):03d}"
        return text

    return cleaned.map(canonical).astype("string")


def parse_number(series):
    raw = series.astype("string").str.strip()
    comma_share = float(raw.str.contains(",", regex=False, na=False).mean())
    if comma_share > 0.05:
        raw = raw.str.replace(".", "", regex=False)
        raw = raw.str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")


def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle:
                sample = handle.read(32768)
            delimiter = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
            return encoding, delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"No fue posible detectar el formato: {path}")


csv_paths = list((INTERIM / "processed_edoh").rglob("*.csv"))
trip_paths = [
    path for path in csv_paths
    if "viaje" in normalize(path.stem) and "etapa" not in normalize(path.stem)
]
if not trip_paths:
    raise FileNotFoundError("No se encontró el módulo CSV de viajes.")
trip_path = sorted(trip_paths, key=lambda path: (len(path.parts), len(path.name)))[0]

encoding, delimiter = csv_format(trip_path)
header = pd.read_csv(trip_path, sep=delimiter, encoding=encoding, nrows=0)
column_map = {normalize(column): column for column in header.columns}
required_columns = ["utam_ori", "utam_des", "fexp_vj", "duracion_min"]
missing_columns = sorted(set(required_columns) - set(column_map))
if missing_columns:
    raise KeyError(f"Faltan columnas: {missing_columns}")

trips = pd.read_csv(
    trip_path,
    sep=delimiter,
    encoding=encoding,
    usecols=[column_map[column] for column in required_columns],
    dtype="string",
    low_memory=False,
)
trips.columns = [normalize(column) for column in trips.columns]
trips["utam_ori"] = canonical_utam(trips["utam_ori"])
trips["utam_des"] = canonical_utam(trips["utam_des"])
trips["weight"] = parse_number(trips["fexp_vj"])
trips["duration_min"] = parse_number(trips["duracion_min"])
trips = trips[
    trips[["utam_ori", "utam_des"]].notna().all(axis=1)
    & trips["weight"].gt(0)
].copy()
trips["valid_duration"] = trips["duration_min"].gt(0)
trips["duration_weight"] = trips["weight"].where(trips["valid_duration"], 0.0)
trips["weighted_duration"] = (
    trips["weight"] * trips["duration_min"].where(trips["valid_duration"])
)

duration_edges = (
    trips.groupby(["utam_ori", "utam_des"], observed=True)
    .agg(
        recomputed_weighted_trips=("weight", "sum"),
        duration_weight=("duration_weight", "sum"),
        weighted_duration=("weighted_duration", "sum"),
    )
    .reset_index()
)
duration_edges["mean_duration_min"] = (
    duration_edges["weighted_duration"]
    / duration_edges["duration_weight"].replace(0, np.nan)
)

edges = pd.read_parquet(EDGE_PATH).copy()
for column in ("utam_ori", "utam_des"):
    edges[column] = canonical_utam(edges[column])
edges = edges.dropna(subset=["utam_ori", "utam_des"]).copy()
edges = edges.merge(duration_edges, on=["utam_ori", "utam_des"], how="left")

relative_weight_error = (
    (edges["recomputed_weighted_trips"] - edges["weighted_trips"]).abs()
    / edges["weighted_trips"].replace(0, np.nan)
)
duration_edge_weight_coverage = float(
    edges.loc[edges["mean_duration_min"].notna(), "weighted_trips"].sum()
    / edges["weighted_trips"].sum()
)

known = edges[edges["mean_duration_min"].notna()].copy()
origin_duration = (
    known.assign(product=known["weighted_trips"] * known["mean_duration_min"])
    .groupby("utam_ori")
    .agg(product=("product", "sum"), weight=("weighted_trips", "sum"))
)
origin_duration["origin_mean_duration"] = origin_duration["product"] / origin_duration["weight"]
edges = edges.merge(
    origin_duration[["origin_mean_duration"]], left_on="utam_ori", right_index=True, how="left"
)
global_mean_duration = float(
    np.average(known["mean_duration_min"], weights=known["weighted_trips"])
)
edges["duration_imputation"] = np.select(
    [
        edges["mean_duration_min"].notna(),
        edges["origin_mean_duration"].notna(),
    ],
    ["observed_edge", "origin_mean"],
    default="global_mean",
)
edges["duration_used_min"] = (
    edges["mean_duration_min"]
    .fillna(edges["origin_mean_duration"])
    .fillna(global_mean_duration)
)

nodes = sorted(set(edges["utam_ori"]) | set(edges["utam_des"]))
node_to_index = {node: index for index, node in enumerate(nodes)}
n_nodes = len(nodes)
P = np.zeros((n_nodes, n_nodes), dtype=float)
D = np.full((n_nodes, n_nodes), np.nan, dtype=float)
W = np.zeros((n_nodes, n_nodes), dtype=float)
for row in edges.itertuples(index=False):
    i = node_to_index[row.utam_ori]
    j = node_to_index[row.utam_des]
    P[i, j] = float(row.transition_probability)
    D[i, j] = float(row.duration_used_min)
    W[i, j] = float(row.weighted_trips)

row_sum_error = float(np.max(np.abs(P.sum(axis=1) - 1.0)))
duration_matrix = np.where(P > 0, D, 0.0)
reward_per_origin = np.sum(P * duration_matrix, axis=1)
outflow = W.sum(axis=1)

duration_audit = {
    "trip_rows_used": int(len(trips)),
    "kernel_edges": int(len(edges)),
    "duration_edge_weight_coverage": duration_edge_weight_coverage,
    "observed_duration_edges": int(edges["mean_duration_min"].notna().sum()),
    "origin_mean_imputed_edges": int(edges["duration_imputation"].eq("origin_mean").sum()),
    "global_mean_imputed_edges": int(edges["duration_imputation"].eq("global_mean").sum()),
    "global_mean_duration_min": global_mean_duration,
    "max_relative_kernel_weight_error": float(relative_weight_error.max()),
    "max_probability_row_sum_error": row_sum_error,
}
edges[[
    "utam_ori", "utam_des", "sampled_trips", "weighted_trips",
    "transition_probability", "mean_duration_min", "duration_used_min",
    "duration_imputation",
]].to_parquet(OUT / "transition_edges_utam_with_duration.parquet", index=False)
pd.Series(duration_audit, name="value").to_csv(OUT / "duration_audit.csv")
display(pd.Series(duration_audit, name="value"))


## 3. Solución de primera llegada y recompensa

Para un conjunto objetivo (A), con (Q=P_{A^c,A^c}), se resuelven

\[
(I-Q)h=\mathbf{1},
\qquad
(I-Q)m_2=\mathbf{1}+2Qh,
\qquad
(I-Q)g=r.
\]

Aquí (h) es el número esperado de transiciones, (m_2) su segundo momento, (g) los minutos acumulados esperados y (r_i=\sum_jP_{ij}d_{ij}) la duración esperada del siguiente paso desde (i).


In [ ]:
def solve_first_passage(P, reward, targets):
    target_mask = np.zeros(P.shape[0], dtype=bool)
    target_mask[targets] = True
    transient = np.flatnonzero(~target_mask)
    Q = P[np.ix_(transient, transient)]
    system = np.eye(len(transient)) - Q
    condition_number = float(np.linalg.cond(system))

    mean = np.linalg.solve(system, np.ones(len(transient)))
    second = np.linalg.solve(system, np.ones(len(transient)) + 2 * Q @ mean)
    variance = np.maximum(second - mean**2, 0.0)
    minutes = np.linalg.solve(system, reward[transient])

    full_mean = np.zeros(P.shape[0])
    full_sd = np.zeros(P.shape[0])
    full_minutes = np.zeros(P.shape[0])
    full_mean[transient] = mean
    full_sd[transient] = np.sqrt(variance)
    full_minutes[transient] = minutes
    return transient, Q, full_mean, full_sd, full_minutes, condition_number


def origin_quantiles(Q, probabilities=(0.50, 0.90, 0.95), horizon=1000):
    survival = np.ones(Q.shape[0])
    result = {probability: np.full(Q.shape[0], -1, dtype=int) for probability in probabilities}
    for step in range(1, horizon + 1):
        survival = Q @ survival
        cdf = 1.0 - survival
        for probability in probabilities:
            unresolved = result[probability] < 0
            crossed = unresolved & (cdf >= probability)
            result[probability][crossed] = step
        if all((values >= 0).all() for values in result.values()):
            break
    return result, survival, int(step)


def aggregate_quantiles(Q, initial, probabilities=(0.50, 0.90, 0.95), horizon=1000):
    distribution = initial.copy()
    result = {}
    for step in range(1, horizon + 1):
        distribution = distribution @ Q
        cdf = 1.0 - float(distribution.sum())
        for probability in probabilities:
            if probability not in result and cdf >= probability:
                result[probability] = int(step)
        if len(result) == len(probabilities):
            break
    return result, float(distribution.sum()), int(step)


target_table = pd.read_csv(TARGET_PATH, dtype={"utam": "string"})
target_table["utam"] = target_table["utam"].astype("string").str.strip().str.upper()
expected_pairs = {(p, d) for p in PURPOSES for d in TARGET_DEFINITIONS}
observed_pairs = set(zip(target_table["purpose_group"], target_table["target_definition"]))
if not expected_pairs.issubset(observed_pairs):
    raise RuntimeError(f"Faltan conjuntos objetivo: {sorted(expected_pairs - observed_pairs)}")

result_rows = []
summary_rows = []
solutions = {}
target_coverage_rows = []
for purpose, definition in sorted(expected_pairs):
    codes = set(target_table.loc[
        target_table["purpose_group"].eq(purpose)
        & target_table["target_definition"].eq(definition),
        "utam",
    ])
    mapped_codes = sorted(codes & set(nodes))
    missing_codes = sorted(codes - set(nodes))
    targets = np.array([node_to_index[code] for code in mapped_codes], dtype=int)
    target_coverage_rows.append({
        "purpose_group": purpose,
        "target_definition": definition,
        "declared_target_nodes": int(len(codes)),
        "mapped_target_nodes": int(len(mapped_codes)),
        "target_node_coverage": float(len(mapped_codes) / len(codes) if codes else 0),
        "missing_codes": ";".join(missing_codes),
    })

    transient, Q, mean, sd, minutes, condition_number = solve_first_passage(
        P, reward_per_origin, targets
    )
    quantiles, tail_survival, horizon_used = origin_quantiles(
        Q, horizon=QUANTILE_HORIZON
    )
    initial = outflow[transient] / outflow[transient].sum()
    aggregate_q, aggregate_tail, aggregate_horizon = aggregate_quantiles(
        Q, initial, horizon=QUANTILE_HORIZON
    )

    full_q = {probability: np.zeros(n_nodes, dtype=int) for probability in quantiles}
    for probability, values in quantiles.items():
        full_q[probability][transient] = values

    target_mask = np.zeros(n_nodes, dtype=bool)
    target_mask[targets] = True
    solutions[(purpose, definition)] = {
        "targets": set(targets.tolist()),
        "mean": mean,
        "minutes": minutes,
    }
    for index, node in enumerate(nodes):
        result_rows.append({
            "purpose_group": purpose,
            "target_definition": definition,
            "utam": node,
            "is_target": bool(target_mask[index]),
            "mean_fpt_steps": float(mean[index]),
            "sd_fpt_steps": float(sd[index]),
            "median_fpt_steps": int(full_q[0.50][index]),
            "p90_fpt_steps": int(full_q[0.90][index]),
            "p95_fpt_steps": int(full_q[0.95][index]),
            "expected_accumulated_minutes": float(minutes[index]),
            "weighted_outflow": float(outflow[index]),
        })

    mixture_mean = float(initial @ mean[transient])
    mixture_second = float(initial @ (sd[transient] ** 2 + mean[transient] ** 2))
    summary_rows.append({
        "purpose_group": purpose,
        "target_definition": definition,
        "target_nodes": int(len(targets)),
        "mean_fpt_steps": mixture_mean,
        "sd_fpt_steps": float(np.sqrt(max(mixture_second - mixture_mean**2, 0))),
        "median_fpt_steps": int(aggregate_q.get(0.50, -1)),
        "p90_fpt_steps": int(aggregate_q.get(0.90, -1)),
        "p95_fpt_steps": int(aggregate_q.get(0.95, -1)),
        "expected_accumulated_minutes": float(initial @ minutes[transient]),
        "condition_number": condition_number,
        "unresolved_origin_quantiles": int(sum((values < 0).sum() for values in quantiles.values())),
        "max_tail_survival_at_horizon": float(tail_survival.max()),
        "aggregate_tail_survival": aggregate_tail,
        "horizon_steps": int(max(horizon_used, aggregate_horizon)),
    })

fpt_results = pd.DataFrame(result_rows)
fpt_summary = pd.DataFrame(summary_rows).sort_values(["purpose_group", "target_definition"])
target_coverage = pd.DataFrame(target_coverage_rows)
fpt_results.to_csv(OUT / "substantive_fpt_by_origin.csv", index=False)
fpt_results.to_parquet(OUT / "substantive_fpt_by_origin.parquet", index=False)
fpt_summary.to_csv(OUT / "substantive_fpt_summary.csv", index=False)
target_coverage.to_csv(OUT / "target_network_coverage.csv", index=False)
display(target_coverage)
display(fpt_summary)


## 4. Sensibilidad `core_50` frente a `extended_80`

La extensión contiene al núcleo por construcción. Por ello, ampliar destinos nunca debería aumentar el tiempo de primera llegada. Además se compara la estabilidad del ordenamiento de las UTAM mediante correlación de Spearman, restringida a orígenes que no pertenecen al conjunto extendido.


In [ ]:
sensitivity_rows = []
for purpose in PURPOSES:
    core = solutions[(purpose, "core_50")]
    extended = solutions[(purpose, "extended_80")]
    nested = core["targets"].issubset(extended["targets"])
    eligible = np.array([
        index for index in range(n_nodes) if index not in extended["targets"]
    ])
    rho_steps = float(spearmanr(
        core["mean"][eligible], extended["mean"][eligible]
    ).statistic)
    rho_minutes = float(spearmanr(
        core["minutes"][eligible], extended["minutes"][eligible]
    ).statistic)
    nonincrease_steps = bool(
        np.all(extended["mean"] <= core["mean"] + 1e-10)
    )
    nonincrease_minutes = bool(
        np.all(extended["minutes"] <= core["minutes"] + 1e-8)
    )
    core_summary = fpt_summary[
        fpt_summary["purpose_group"].eq(purpose)
        & fpt_summary["target_definition"].eq("core_50")
    ].iloc[0]
    extended_summary = fpt_summary[
        fpt_summary["purpose_group"].eq(purpose)
        & fpt_summary["target_definition"].eq("extended_80")
    ].iloc[0]
    sensitivity_rows.append({
        "purpose_group": purpose,
        "nested_targets": bool(nested),
        "eligible_non_target_origins": int(len(eligible)),
        "spearman_steps": rho_steps,
        "spearman_minutes": rho_minutes,
        "extended_nonincrease_steps": nonincrease_steps,
        "extended_nonincrease_minutes": nonincrease_minutes,
        "core_to_extended_mean_steps_ratio": float(
            core_summary["mean_fpt_steps"] / extended_summary["mean_fpt_steps"]
        ),
        "core_to_extended_mean_minutes_ratio": float(
            core_summary["expected_accumulated_minutes"]
            / extended_summary["expected_accumulated_minutes"]
        ),
    })

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUT / "target_definition_sensitivity.csv", index=False)
display(sensitivity)


## 5. Resultados gráficos

Los paneles comparan accesibilidad promedio y sensibilidad de las UTAM que permanecen fuera del conjunto extendido.


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

sns.barplot(
    data=fpt_summary,
    x="purpose_group",
    y="mean_fpt_steps",
    hue="target_definition",
    ax=axes[0],
)
axes[0].set_xlabel("Propósito")
axes[0].set_ylabel("Primera llegada media (transiciones)")

sns.barplot(
    data=fpt_summary,
    x="purpose_group",
    y="expected_accumulated_minutes",
    hue="target_definition",
    ax=axes[1],
)
axes[1].set_xlabel("Propósito")
axes[1].set_ylabel("Minutos acumulados esperados")

palette = {"employment": "#4C72B0", "education": "#55A868", "health": "#C44E52"}
for purpose in PURPOSES:
    core = solutions[(purpose, "core_50")]
    extended = solutions[(purpose, "extended_80")]
    eligible = np.array([
        index for index in range(n_nodes) if index not in extended["targets"]
    ])
    axes[2].scatter(
        core["mean"][eligible],
        extended["mean"][eligible],
        s=22,
        alpha=0.7,
        label=purpose,
        color=palette[purpose],
    )
limit = max(
    float(fpt_results["mean_fpt_steps"].max()), 1.0
)
axes[2].plot([0, limit], [0, limit], linestyle="--", color="black", linewidth=1)
axes[2].set_xlabel("FPT con núcleo 50 %")
axes[2].set_ylabel("FPT con extensión 80 %")
axes[2].legend(title="Propósito")

fig.tight_layout()
fig.savefig(OUT / "substantive_fpt_summary.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Compuerta 7

La compuerta exige consistencia entre el kernel y los microdatos, cobertura temporal suficiente, objetivos completamente representados, soluciones finitas y la monotonía matemática esperada al ampliar el conjunto objetivo.


In [ ]:
thresholds = {
    "minimum_duration_edge_weight_coverage": 0.95,
    "maximum_relative_kernel_weight_error": 1e-8,
    "maximum_probability_row_sum_error": 1e-10,
    "minimum_target_node_coverage": 0.99,
    "maximum_condition_number": 1e6,
    "minimum_spearman_steps": 0.50,
}

checks = {
    "gate5_dependency_valid": bool(gate5_status.get("gate_5_passed", False)),
    "gate6_dependency_valid": bool(gate6_status.get("gate_6_passed", False)),
    "duration_coverage_valid": bool(
        duration_edge_weight_coverage
        >= thresholds["minimum_duration_edge_weight_coverage"]
    ),
    "kernel_weight_consistency_valid": bool(
        relative_weight_error.max()
        <= thresholds["maximum_relative_kernel_weight_error"]
    ),
    "kernel_probability_valid": bool(
        row_sum_error <= thresholds["maximum_probability_row_sum_error"]
    ),
    "target_coverage_valid": bool(
        target_coverage["target_node_coverage"].ge(
            thresholds["minimum_target_node_coverage"]
        ).all()
    ),
    "finite_solutions_valid": bool(
        np.isfinite(fpt_results[[
            "mean_fpt_steps", "sd_fpt_steps", "expected_accumulated_minutes"
        ]].to_numpy()).all()
    ),
    "quantiles_resolved_valid": bool(
        (fpt_summary["unresolved_origin_quantiles"] == 0).all()
    ),
    "conditioning_valid": bool(
        fpt_summary["condition_number"].max()
        <= thresholds["maximum_condition_number"]
    ),
    "nested_targets_valid": bool(sensitivity["nested_targets"].all()),
    "monotonicity_steps_valid": bool(
        sensitivity["extended_nonincrease_steps"].all()
    ),
    "monotonicity_minutes_valid": bool(
        sensitivity["extended_nonincrease_minutes"].all()
    ),
    "ranking_sensitivity_valid": bool(
        sensitivity["spearman_steps"].ge(
            thresholds["minimum_spearman_steps"]
        ).all()
    ),
}
checks["gate_7_passed"] = bool(all(checks.values()))

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "source_sha256": str(source_manifest.loc[0, "sha256"]),
    "model": "absorbing Markov chain with edge-duration rewards",
    "scope": "survey-revealed destination opportunities",
    "thresholds": {key: float(value) for key, value in thresholds.items()},
    "duration_audit": duration_audit,
    "target_coverage": json.loads(target_coverage.to_json(orient="records")),
    "fpt_summary": json.loads(fpt_summary.to_json(orient="records")),
    "sensitivity": json.loads(sensitivity.to_json(orient="records")),
    "gate": checks,
}
(OUT / "gate_7.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
pd.Series(checks, name="result").to_csv(OUT / "gate_7_checks.csv")
display(pd.Series(checks, name="result"))

persistent_out = DRIVE_BASE / "results/gate7"
persistent_out.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent_out / artifact.name)

print("Compuerta 7:", "APROBADA" if checks["gate_7_passed"] else "NO APROBADA")
print("Copia persistente:", persistent_out)


## Interpretación y continuación

- Los resultados describen accesibilidad potencial bajo el flujo agregado observado; no son trayectorias personales ni efectos causales.
- La comparación entre `core_50` y `extended_80` es parte del resultado y debe conservarse, no utilizarse para seleccionar la definición más favorable.
- Si la Compuerta 7 pasa, el siguiente notebook estimará heterogeneidad por sexo y estrato mediante kernels regularizados hacia el kernel poblacional y bootstrap de hogares o personas.
- La cartografía y la validación con establecimientos abiertos se añadirán como análisis separados para no confundir demanda revelada con oferta institucional.
